In [1]:
from pgmpy.models import LinearGaussianBayesianNetwork
from sklearn.model_selection import train_test_split
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [2]:
def generate_dag_structure(
    n_nodes: int,
    max_in_degree: int,
    p_extra: float = 0.0,
    ensure_connected: bool = True,
    seed: int | None = None
):
    rng = random.Random(seed)
    nodes = [f"X_{i}" for i in range(n_nodes)]

    # Random topological order
    order = nodes[:]
    rng.shuffle(order)
    position = {node: i for i, node in enumerate(order)}

    edges = []
    parents = {node: set() for node in nodes}

    # Backbone for weak connectivity
    if ensure_connected:
        for i in range(1, n_nodes):
            child = order[i]
            parent = rng.choice(order[:i])
            parents[child].add(parent)
            edges.append((parent, child))

    # Extra edges
    for child in order:
        possible_parents = [
            p for p in order[:position[child]]
            if p not in parents[child]
        ]
        rng.shuffle(possible_parents)
        for parent in possible_parents:
            if len(parents[child]) >= max_in_degree:
                break
            if rng.random() < p_extra:
                parents[child].add(parent)
                edges.append((parent, child))

    return edges

# ----------------------------
# Generate 10 unique DAGs
# ----------------------------
def generate_unique_gbns(n_dags=10, n_nodes=8, max_in_degree=2, p_extra_range=(0.2, 0.4)):
    """
    Generate `n_dags` unique GBNs, each with its own random p_extra sampled from p_extra_range.
    """
    unique_edges = set()
    gbns = []
    seed = 0

    while len(gbns) < n_dags:
        # Sample a p_extra for this GBN
        p_extra = random.uniform(*p_extra_range)

        edges = generate_dag_structure(
            n_nodes=n_nodes,
            max_in_degree=max_in_degree,
            p_extra=p_extra,
            ensure_connected=True,
            seed=seed
        )

        # Sort edges to standardize representation
        edges_tuple = tuple(sorted(edges))

        if edges_tuple not in unique_edges:
            unique_edges.add(edges_tuple)

            # Create pgmpy GBN
            gbn = LinearGaussianBayesianNetwork()
            node_names = [f"X_{i}" for i in range(n_nodes)]
            gbn.add_nodes_from(node_names)
            gbn.add_edges_from(edges)

            # Save the p_extra for this GBN
            gbn.p_extra = p_extra

            gbns.append(gbn)

        seed += 1  # change seed each iteration

    return gbns


In [3]:
def split_data(data_frame, train_frac, val_frac, test_frac, seed):
    train_val_data, test_data = train_test_split(data_frame, test_size=test_frac, random_state=seed)
    val_size = val_frac / (train_frac + val_frac)
    train_data, val_data = train_test_split(train_val_data, test_size=val_size, random_state=seed)
    return train_data.reset_index(drop=True), val_data.reset_index(drop=True), test_data.reset_index(drop=True)

In [4]:
def train_baseline(true_gbn, train_data):
    gbn_copy = true_gbn.copy()
    gbn_copy.fit(train_data)
    nodes = list(gbn_copy.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    parents = []
    
    for node in nodes:
        parent_names = true_gbn.get_parents(node)
        parent_indices = [node_to_idx[p] for p in parent_names]
        parents.append(parent_indices)
    
    return gbn_copy, parents, nodes

In [5]:
def extract_gbn_parameters(gbn, nodes, min_std=1e-3):
    """
    Extract parents, intercepts, weights, and stds
    from a Linear Gaussian Bayesian Network.
    """

    parents_dict = {}
    mu_true = []
    weights_true = []
    sigma_true = []

    for node in nodes:
        parents = list(gbn.get_parents(node))
        parents_dict[node] = parents

        cpd = gbn.get_cpds(node)

        # Intercept
        mu_true.append(float(cpd.beta[0]))

        # Weights
        if len(cpd.beta) > 1:
            w = np.array(cpd.beta[1:].flatten(), dtype=float)
        else:
            w = np.array([])
        weights_true.append(w)

        # Std (with lower bound)
        sigma_true.append(max(float(cpd.std), min_std))

    return parents_dict, mu_true, weights_true, sigma_true


In [6]:
# =========================================
# Node-wise NN without dropout
# =========================================
class NodeNN(nn.Module):
    def __init__(self, n_parents, hidden_dims, activation):
        super().__init__()

        input_dim = n_parents if n_parents > 0 else 1
        dims = [input_dim] + hidden_dims

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if activation is not None:
                layers.append(activation())

        self.feature_extractor = nn.Sequential(*layers) if layers else nn.Identity()

        final_dim = dims[-1] if dims else input_dim
        self.mean_out = nn.Linear(final_dim, 1)
        self.logvar_out = nn.Linear(final_dim, 1)

    def forward(self, x):
        x = self.feature_extractor(x)
        mu = self.mean_out(x).squeeze(-1)
        log_var = self.logvar_out(x).squeeze(-1)
        return mu, log_var


# =========================================
# Gaussian NLL
# =========================================
""" def gaussian_nll(mu, log_var, target):
    log_var = torch.clamp(log_var, min=-10.0, max=10.0)  # prevent extreme values
    var = torch.exp(log_var)
    return 0.5 * torch.mean(log_var + (target - mu) ** 2 / var) """


def gaussian_nll(mu, log_var, target):
    log_var = torch.clamp(log_var, min=-10.0)
    var = torch.exp(log_var)
    device = mu.device  # ensure same device as input
    return 0.5 * torch.mean(torch.log(torch.tensor(2 * torch.pi, device=device)) + log_var + ((target - mu)**2)/var)




# =========================================
# Train node-wise NNs
# =========================================
def train_nodewise_nns(
    df_train,
    df_val,
    parents,
    nodes,
    hidden_dims,
    activation,
    device="cpu",
    n_epochs=500,
    lr=1e-3,
    patience=20
):
    nn_models = []
    optimizers = []
    node_train_times = []
    # Create models
    for node in nodes:
        model = NodeNN(
            n_parents=len(parents[node]),
            hidden_dims=hidden_dims,
            activation=activation
        ).to(device)
        nn_models.append(model)
        optimizers.append(torch.optim.Adam(model.parameters(), lr=lr))

    # Early stopping state
    best_val_losses = [float("inf")] * len(nodes)
    patience_counters = [0] * len(nodes)
    best_states = [None] * len(nodes)
    stopped = [False] * len(nodes)

    # Training loop
    for epoch in range(n_epochs):
        epoch_train_loss = 0.0
        active_nodes = 0

        for i, node in enumerate(nodes):
            if stopped[i]:
                continue
            
            start_time = time.time()
            model = nn_models[i]
            optimizer = optimizers[i]
            model.train()

            # TRAIN
            x_train = torch.tensor(df_train[parents[node]].values, dtype=torch.float32, device=device) \
                if parents[node] else torch.zeros((len(df_train), 1), dtype=torch.float32, device=device)
            y_train = torch.tensor(df_train[node].values, dtype=torch.float32, device=device)

            optimizer.zero_grad()
            mu_pred, log_var_pred = model(x_train)
            train_loss = gaussian_nll(mu_pred, log_var_pred, y_train)
            train_loss.backward()
            optimizer.step()

            epoch_train_loss += train_loss.item()
            active_nodes += 1

            end_time = time.time()

            if len(node_train_times) <= i:
                node_train_times.append(end_time - start_time)
            else:
                node_train_times[i] += end_time - start_time

            # VALIDATION
            model.eval()
            with torch.no_grad():
                x_val = torch.tensor(df_val[parents[node]].values, dtype=torch.float32, device=device) \
                    if parents[node] else torch.zeros((len(df_val), 1), dtype=torch.float32, device=device)
                y_val = torch.tensor(df_val[node].values, dtype=torch.float32, device=device)

                mu_val, log_var_val = model(x_val)
                val_loss = gaussian_nll(mu_val, log_var_val, y_val).item()

            # EARLY STOPPING
            if val_loss < best_val_losses[i]:
                best_val_losses[i] = val_loss
                patience_counters[i] = 0
                best_states[i] = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_counters[i] += 1
                if patience_counters[i] >= patience:
                    stopped[i] = True
                    model.load_state_dict(best_states[i])
                    print(f"Node {node} early stopped at epoch {epoch+1}. Best val loss={best_val_losses[i]:.4f}")

        if (epoch + 1) % 50 == 0 and active_nodes > 0:
            print(f"Epoch {epoch+1}, Avg Train Loss: {epoch_train_loss / active_nodes:.4f}")

        if all(stopped):
            print(f"All nodes early-stopped at epoch {epoch+1}.")
            break

    return nn_models, best_val_losses, node_train_times


In [8]:
def baseline_conditional_gaussian_kl(
    true_mu: torch.Tensor,          # scalar
    true_weights: torch.Tensor,     # (n_parents,)
    true_sigma: torch.Tensor,       # scalar
    base_mu: torch.Tensor,          # scalar
    base_weights: torch.Tensor,     # (n_parents,)
    base_sigma: torch.Tensor,       # scalar
    x: torch.Tensor                 # (N, n_parents)
):
    """
    Computes KL( p(y|x) || q_base(y|x) ) averaged over samples.

    p(y|x)      = N(true_mu + x @ true_weights, true_sigma^2)
    q_base(y|x) = N(base_mu + x @ base_weights, base_sigma^2)
    """

    # Conditional means
    mu_true_cond = true_mu + x @ true_weights
    mu_base_cond = base_mu + x @ base_weights

    # Variances
    var_true = true_sigma ** 2
    var_base = base_sigma ** 2

    # KL per sample
    kl_per_sample = (
        0.5 * (
            torch.log(var_base / var_true)
            + (var_true + (mu_true_cond - mu_base_cond) ** 2) / var_base
            - 1.0
        )
    )

    return kl_per_sample.mean()



def conditional_gaussian_nn_kl(
    true_mu: torch.Tensor,          # scalar
    true_weights: torch.Tensor,     # (n_parents,)
    true_sigma: torch.Tensor,       # scalar
    x: torch.Tensor,                # (N, n_parents)
    pred_mu: torch.Tensor,          # (N,)
    pred_log_var: torch.Tensor      # (N,)
):
    """
    Computes KL( p(y|x) || q(y|x) ) averaged over samples.

    p(y|x) = N(true_mu + x @ true_weights, true_sigma^2)
    q(y|x) = N(pred_mu(x), exp(pred_log_var(x)))
    """

    # True conditional mean
    mu_true_cond = true_mu + x @ true_weights

    # Variances
    var_true = true_sigma ** 2
    var_pred = torch.exp(pred_log_var)

    # KL per sample
    kl_per_sample = (
        0.5 * (
            torch.log(var_pred / var_true)
            + (var_true + (mu_true_cond - pred_mu) ** 2) / var_pred
            - 1.0
        )
    )

    return kl_per_sample.mean()


In [9]:


def save_gbn_parameters(gbn, node_names, p_extra, save_path, train_time_sec=None):
    """
    Save GBN structure + parameters in JSON-serialisable form.
    """

    parents, mu, weights, sigma = extract_gbn_parameters(gbn, node_names)

    # ---- Make JSON-safe ----
    mu_json = [float(m) for m in mu]
    sigma_json = [float(s) for s in sigma]

    weights_json = []
    for w in weights:
        if isinstance(w, np.ndarray):
            weights_json.append(w.tolist())
        else:
            weights_json.append([])

    gbn_dict = {
        "nodes": list(node_names),
        "parents": parents,                 # already strings → OK
        "p_extra": p_extra,
        "mu": mu_json,
        "sigma": sigma_json,
        "weights": weights_json,
        "edges": [(str(u), str(v)) for u, v in gbn.edges()],
        "train_time_sec": float(train_time_sec) if train_time_sec is not None else None
    }

    with open(save_path, "w") as f:
        json.dump(gbn_dict, f, indent=2)


def save_nn_predictions(nn_models, node_names, parents_dict, test_data, node_train_times, save_path, device="cpu", hidden_dims=None, activation=None):
    """
    Save node-wise NN predicted mu, sigma (exp(log_var)), and training time on test_data.
    """
    mu_pred_list = []
    sigma_pred_list = []

    for idx, node in enumerate(node_names):
        model = nn_models[idx]
        model.eval()
        if parents_dict[node]:
            x_input = torch.tensor(test_data[parents_dict[node]].values, dtype=torch.float32, device=device)
        else:
            x_input = torch.zeros((len(test_data), 1), dtype=torch.float32, device=device)

        with torch.no_grad():
            mu_pred, log_var_pred = model(x_input)
            mu_pred_list.append(mu_pred.cpu().numpy().tolist())
            sigma_pred_list.append(torch.exp(log_var_pred).cpu().numpy().tolist())

    nn_dict = {
        "nodes": list(node_names),
        "parents": parents_dict,            # node → list of parent names
        "mu_pred": mu_pred_list,
        "sigma_pred": sigma_pred_list,
        "train_time_sec": node_train_times  # <-- per node
    }

    # Optional metadata
    if hidden_dims is not None:
        nn_dict["hidden_dims"] = hidden_dims
    if activation is not None:
        nn_dict["activation_func"] = "linear" if activation is None else activation.__name__

    with open(save_path, "w") as f:
        json.dump(nn_dict, f, indent=2)




In [10]:
experiment_configs = [
    (10, 2),
    (10, 5),
    (20, 2),
    (20, 5),
    (20, 10),
    (50, 2),
    (50, 5),
    (50, 10),
]
sim = 5
sample_size = [200, 600, 1000, 1400]
n_samples = 2000
hidden_dims = [[16], [32], [64], [16,16,16], [32,32,32], [64,64,64], [16,16,16,16,16,16,16,16,16,16], 
               [32,32,32,32,32,32,32,32,32,32], [64,64,64,64,64,64,64,64,64,64], [32,16], [64,32,16], [32,16,32], [64,32,16,32,64]]
act_func = [nn.ReLU, nn.Tanh, nn.Sigmoid]
records = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_RESULTS_DIR = "21-01_Test_Final_Results"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)
print(f"{device}")

for n_nodes_curr, in_deg in experiment_configs:
    node_dir = os.path.join(BASE_RESULTS_DIR, f"GBN_{n_nodes_curr}")
    os.makedirs(node_dir, exist_ok=True)

    gbn_list = generate_unique_gbns(
        n_dags=sim,
        n_nodes=n_nodes_curr,
        max_in_degree=in_deg,
        p_extra_range=(0.05, 0.15)
    )
    in_deg_dir = os.path.join(node_dir, f"GBN_{n_nodes_curr}_indegree_{in_deg}")
    os.makedirs(in_deg_dir, exist_ok=True)

    for i_gbn, gbn in enumerate(gbn_list, 1):
        print(f"========================================================== GBN {i_gbn} ==========================================================")

        gbn_dir = os.path.join(in_deg_dir, f"{i_gbn}. GBN_{n_nodes_curr}_indegree_{in_deg}")
        os.makedirs(gbn_dir, exist_ok=True)
        gbn.get_random_cpds(inplace=True)

        gbn_json_path = os.path.join(gbn_dir, "true_gbn.json")
        node_names = sorted(gbn.nodes(), key=lambda x: int(x.split("_")[1]))
        save_gbn_parameters(gbn, node_names, gbn.p_extra, gbn_json_path)

        for samples in sample_size:
            print(f"========================================================== Sample Size {samples} ==========================================================")

            data = gbn.simulate(samples)
            train_data, val_data, test_data = split_data(data, 0.6, 0.2, 0.2, 42)

            exp_dir = os.path.join(gbn_dir, f"{i_gbn}. GBN_{n_nodes_curr}_indegree_{in_deg}_samples_{samples}")
            os.makedirs(exp_dir, exist_ok=True)
            exp_results = []

            train_data.to_csv(os.path.join(exp_dir, "train.csv"), index=False)
            val_data.to_csv(os.path.join(exp_dir, "val.csv"), index=False)
            test_data.to_csv(os.path.join(exp_dir, "test.csv"), index=False)

            baseline_start = time.perf_counter()
            baseline_gbn, parents_list, baseline_nodes = train_baseline(gbn, train_data)
            baseline_train_time_sec = time.perf_counter() - baseline_start

            save_gbn_parameters(
                baseline_gbn,
                node_names,
                gbn.p_extra,
                os.path.join(exp_dir, "baseline_gbn.json"),
                train_time_sec=baseline_train_time_sec
            )

            node_names = sorted(gbn.nodes(), key=lambda x: int(x.split("_")[1]))
            true_parents_dict, true_mu, true_weights, true_sigma = extract_gbn_parameters(gbn, node_names)
            baseline_parents_dict, baseline_mu, baseline_weights, baseline_sigma = extract_gbn_parameters(baseline_gbn, node_names)

            kl_base_node_list = []
            for idx, node in enumerate(node_names):
                if true_parents_dict[node]:
                    parent_cols = true_parents_dict[node]
                    x_input = torch.tensor(
                        test_data.loc[:, parent_cols].values,
                        dtype=torch.float32, device=device
                    )
                    w_t = torch.tensor(true_weights[idx], dtype=torch.float32, device=device)
                    w_b = torch.tensor(baseline_weights[idx], dtype=torch.float32, device=device)
                else:
                    x_input = torch.zeros((test_data.shape[0], 1), dtype=torch.float32, device=device)
                    w_t = torch.zeros(1, dtype=torch.float32, device=device)
                    w_b = torch.zeros(1, dtype=torch.float32, device=device)

                mu_t = torch.tensor(true_mu[idx], dtype=torch.float32, device=device)
                sigma_t = torch.tensor(true_sigma[idx], dtype=torch.float32, device=device)
                mu_b = torch.tensor(baseline_mu[idx], dtype=torch.float32, device=device)
                sigma_b = torch.tensor(baseline_sigma[idx], dtype=torch.float32, device=device)

                kl_node = baseline_conditional_gaussian_kl(mu_t, w_t, sigma_t, mu_b, w_b, sigma_b, x_input)
                kl_base_node_list.append(kl_node.item())

            kl_baseline_avg = sum(kl_base_node_list) / len(kl_base_node_list)

            for dim in hidden_dims:
                print(f"========================================================== Hidden Dim {dim} ==========================================================")
                for activation in act_func:
                    print(f"========================================================== Activation Function {activation} ==========================================================")

                    nn_models, best_val_losses, node_train_times = train_nodewise_nns(
                        train_data, val_data,
                        true_parents_dict, node_names,
                        dim, activation,
                        device=device, n_epochs=500, lr=1e-3, patience=20
                    )

                    kl_nn_node_list = []
                    for idx, model in enumerate(nn_models):
                        node_curr = node_names[idx]
                        model.eval()

                        if true_parents_dict[node_curr]:
                            parent_cols = true_parents_dict[node_curr]
                            x_input = torch.tensor(
                                test_data.loc[:, parent_cols].values,
                                dtype=torch.float32, device=device
                            )
                            w_t = torch.tensor(true_weights[idx], dtype=torch.float32, device=device)
                        else:
                            x_input = torch.zeros((test_data.shape[0], 1), dtype=torch.float32, device=device)
                            w_t = torch.zeros(1, dtype=torch.float32, device=device)

                        mu_t = torch.tensor(true_mu[idx], dtype=torch.float32, device=device)
                        sigma_t = torch.tensor(true_sigma[idx], dtype=torch.float32, device=device)

                        with torch.no_grad():
                            mu_pred, log_var_pred = model(x_input)
                            mu_pred = mu_pred.squeeze()
                            log_var_pred = log_var_pred.squeeze()

                        kl_node = conditional_gaussian_nn_kl(mu_t, w_t, sigma_t, x_input, mu_pred, log_var_pred)
                        kl_nn_node_list.append(kl_node.item())

                    kl_nn_avg = sum(kl_nn_node_list) / len(kl_nn_node_list)

                    nn_pred_path = os.path.join(exp_dir, f"nn_pred_{dim}_{activation.__name__}.json")
                    save_nn_predictions(
                        nn_models, node_names, true_parents_dict, test_data,
                        node_train_times, nn_pred_path, device=device,
                        hidden_dims=dim, activation=activation
                    )

                    exp_results.append({
                        "n_nodes": n_nodes_curr,
                        "max_in_degree": in_deg,
                        "n_samples": samples,
                        "hidden_dims": str(dim),
                        "activation_func": "linear" if activation is None else activation.__name__,
                        "baseline_kl_avg": kl_baseline_avg,
                        "nn_kl_avg": kl_nn_avg,
                        "nn_val_nll_mean": np.mean(best_val_losses),
                        "nn_val_nll_std": np.std(best_val_losses),
                        "nn_train_time_mean": np.mean(node_train_times),
                        "mle_train_time_sec": baseline_train_time_sec
                    })

            results_df = pd.DataFrame(exp_results)
            results_df.to_csv(os.path.join(exp_dir, "results.csv"), index=False)

cuda
========================================================== GBN 1 ==========================================================
========================================================== Sample Size 200 ==========================================================
========================================================== Hidden Dim [16] ==========================================================
========================================================== Activation Function <class 'torch.nn.modules.activation.ReLU'> ==========================================================
Node X_10 early stopped at epoch 50. Best val loss=1.3478
Epoch 50, Avg Train Loss: 1587.5048
Node X_1 early stopped at epoch 77. Best val loss=1.2567
Epoch 100, Avg Train Loss: 621.3596
Node X_36 early stopped at epoch 129. Best val loss=1.0010
Epoch 150, Avg Train Loss: 477.7288
Epoch 200, Avg Train Loss: 385.6620
Node X_0 early stopped at epoch 220. Best val loss=1.2697
Node X_12 early stopped at epoch 239. Best val

In [3]:
import json
import re
import pandas as pd
from pathlib import Path

base = Path("21-01_Test_Final_Results")

rows = []

for fp in base.rglob("nn_pred_*.json"):
    with open(fp, "r") as f:
        d = json.load(f)

    full_path = str(fp)

    # graph size from folder name like GBN_10
    nodes_match = re.search(r"GBN_(\d+)", full_path)

    # in-degree from path like indegree_2 or indeg_2
    indegree_match = re.search(r"indeg(?:ree)?_(\d+)", full_path, re.IGNORECASE)

    # hidden dims from filename like nn_pred_[16, 16]_ReLU
    hidden_match = re.search(r"nn_pred_(\[[^\]]+\])", fp.stem)
    act_match = re.search(r"_(ReLU|Tanh|Sigmoid)$", fp.stem)

    train_times = d.get("train_time_sec", [])

    rows.append({
        "file": full_path,
        "n_nodes": int(nodes_match.group(1)) if nodes_match else None,
        "max_in_degree": int(indegree_match.group(1)) if indegree_match else None,
        "hidden_dims": hidden_match.group(1) if hidden_match else None,
        "activation": act_match.group(1) if act_match else None,
        "gbn_train_time_sec": sum(train_times) if train_times else None,
        "avg_node_train_time_sec": (sum(train_times) / len(train_times)) if train_times else None,
    })

df_times = pd.DataFrame(rows)
print(df_times.head())

                                                file  n_nodes  max_in_degree  \
0  21-01_Test_Final_Results\GBN_10\GBN_10_indegre...       10              2   
1  21-01_Test_Final_Results\GBN_10\GBN_10_indegre...       10              2   
2  21-01_Test_Final_Results\GBN_10\GBN_10_indegre...       10              2   
3  21-01_Test_Final_Results\GBN_10\GBN_10_indegre...       10              2   
4  21-01_Test_Final_Results\GBN_10\GBN_10_indegre...       10              2   

                                hidden_dims activation  gbn_train_time_sec  \
0  [16, 16, 16, 16, 16, 16, 16, 16, 16, 16]       ReLU           13.052867   
1  [16, 16, 16, 16, 16, 16, 16, 16, 16, 16]    Sigmoid           11.350999   
2  [16, 16, 16, 16, 16, 16, 16, 16, 16, 16]       Tanh           14.437873   
3                              [16, 16, 16]       ReLU            8.836792   
4                              [16, 16, 16]    Sigmoid           13.797538   

   avg_node_train_time_sec  
0                 1.3

In [4]:
df_times.to_csv("NN_times.csv")